In [1]:
import glob
import os
from pathlib import Path
import pandas as pd
import transformers.utils.hub
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, PeftModel
import argparse
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer
from utils.seeds import initialize_seeds
from utils.prompts import get_system_prompt
import json


os.environ["TOKENIZERS_PARALLELISM"] = "false"

personas = json.load(open("./golden-dataset/personas_desc.json", "r"))

def preprocess_dpo(example):
    persona = example["persona"]
    description = personas[persona]
    return {
        "prompt": [{"role": "system", "content": get_system_prompt(persona, description)},
                   {"role": "user", "content": example["prompt"]}],
        "chosen": [{"role": "assistant", "content": example["preferred_response"]}],
        "rejected": [{"role": "assistant", "content": example["rejected_response"]}],
    }

def preprocess_sft(example):
    persona = example["persona"]
    description = personas[persona]
    return {
        "prompt": [{"role": "system", "content": get_system_prompt(persona, description)},
                   {"role": "user", "content": example["prompt"]}],
        "completion": [{"role": "assistant", "content": example["preferred_response"]}],
    }

def find_resume_checkpoint(output_dir):
    """Return the path to the last checkpoint in output_dir, or None if there isn't one."""
    if not os.path.isdir(output_dir):
        return None
    last_checkpoint = get_last_checkpoint(output_dir)
    if last_checkpoint is not None:
        print(f"Found existing checkpoint at {last_checkpoint}, will resume from there.")
    return last_checkpoint


def main(model, batch_size=16, grad_accumulation_steps=1):
    initialize_seeds()

    model_name = model.split("/")[-1]


    print(f"Training model {model_name} with SFT + DPO")
    
    print("=== LOADING DATASET ===")
    train_data = load_dataset("json", data_files="golden-dataset/train_clean.jsonl")

    print("=== PREPROCESSING DATASET ===")
    sft_dataset = train_data.map(preprocess_sft, remove_columns=["persona", "query_type", "preferred_response", "rejected_response"])["train"]
    dpo_dataset = train_data.map(preprocess_dpo, remove_columns=["persona", "query_type", "preferred_response", "rejected_response"])["train"]

    sft_output_dir = f"models/{model_name}-SFT"
    dpo_output_dir = f"models/{model_name}-SFT+DPO"
    print(sft_dataset, dpo_dataset)

    peft_config = LoraConfig(
        r=16,
        lora_alpha=16,
        lora_dropout=0.0,
        task_type="CAUSAL_LM",
        target_modules= ["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
    )

    sft_config = SFTConfig(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accumulation_steps,
        warmup_steps=.1,
        num_train_epochs=3,
        learning_rate=2e-4,
        weight_decay=0.01,
        logging_strategy="epoch",
        lr_scheduler_type="linear",
        seed=42,
        output_dir=sft_output_dir,
        save_strategy="epoch",
    )


    dpo_config = DPOConfig(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accumulation_steps,
        warmup_steps=.1,
        num_train_epochs=3,
        learning_rate=5e-6,
        weight_decay=0.01,
        logging_strategy="epoch",
        lr_scheduler_type="linear",
        seed=42,
        beta=0.1,
        output_dir=dpo_output_dir,
        save_strategy="epoch",

_IncompleteInputError: incomplete input (1355117960.py, line 105)

In [1]:
from datasets import load_dataset
from itertools import islice


/home/peluz/projects/univie/Safe-Actor/safe-actor/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
ds = load_dataset(
        "proj-persona/PersonaHub",
        "elite_persona",
        split="train",
        streaming=True,   # <-- key part: no full download, reads records lazily
    )


In [ ]:
ds = ds.select_columns(["persona"])

In [10]:
ds = ds.shuffle(seed=42, buffer_size=1000)

In [11]:
samples = list(islice(ds, 400))

In [14]:
import json

In [16]:
json.dump([x["persona"] for x in samples], open("data/personaHub.json", "w"), indent=True)

In [ ]:
def sample_personas(n=10, subset="elite_persona", seed=42, shuffle_buffer=1_000):
    """
    Sample n personas from PersonaHub without downloading the full dataset.

    subset options include (check the dataset repo for the current list):
    'persona', 'elite_persona', 'instruction', 'knowledge',
    'math', 'npc', 'reasoning', 'tool', ...
    """
    ds = load_dataset(
        "proj-persona/PersonaHub",
        subset,
        split="train",
        streaming=True,   # <-- key part: no full download, reads records lazily
    )

    # Shuffle with a reservoir buffer (approximate shuffling for streaming datasets)
    ds = ds.shuffle(seed=seed, buffer_size=shuffle_buffer)

    # Pull just n examples
    samples = list(islice(ds, n))
    return samples

In [ ]:
sample_personas()

In [1]:
DATA_DIR = "./golden-dataset/data/"
files = glob.glob(f"{DATA_DIR}*_dataset.csv")
files = sorted(files)
dataset = "xstest"
personaHub = True

if dataset == "xstest":
    df_list = [pd.read_csv(f) for f in files if "empty" not in f and "refused" not in f]
    merged_df = pd.concat(df_list, axis=0, ignore_index=True)
    test_df = merged_df[merged_df["split"] == "test"]
    suffix = ""
if not personaHub:
    with open('./golden-dataset/personas_desc.json', 'r') as f:
        personas = json.load(f)
else:
    personas = {"personaHub": None}
    suffix = "_personaHub"

    

for persona_name, persona_desc in personas.items():
    persona_instruction = f"You are exactly this character: {persona_name}. {persona_desc}"
    responses = []
    print(
            f"Generating responses for role {persona_name} from model {model}"
        )
    if dataset == "xstest" and not personaHub:
        queries = test_df[test_df['persona'] == persona_name]["prompt"]
        prompts = [[
                    {"role": "system", "content": persona_instruction},
                    {"role": "user", "content": q}]
                    for q in queries]
    elif dataset == "xstest" and personaHub:
        queries = test_df[test_df.persona.str.contains("baker", case=False)]["prompt"]
        try:
            hub_personas = json.load(open("data/pesonaHub.json", "r"))
        except FileNotFoundError:
            hub_personas = [x["persona"] for x in sample_personas(n = len(queries))]
            json.dump(hub_personas, open("data/personaHub.json", "w"), indent=True)
        prompts = [[
                    {"role": "system", "content": f"You are exactly this character: {p}"},
                    {"role": "user", "content": q}]
                    for q, p in zip(queries, hub_personas)]
    print(f"Example prompt: {prompts[0]}")

NameError: name 'glob' is not defined